In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import os

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

2026-04-28 19:13:38.237782: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777403618.616925      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777403618.720666      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777403619.690348      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777403619.690399      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777403619.690403      55 computation_placer.cc:177] computation placer alr

In [2]:
CHECKPOINT_FILEPATH = '/kaggle/working/chkpts/checkpoint.model.keras'

In [3]:
model = EfficientNetB0(
    weights='imagenet', 
    include_top=False, 
    input_shape=(224, 224, 3)
)

# freeze weights
model.trainable = False

inputs = keras.Input(shape=(224, 224, 3))

x = model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)

outputs = layers.Dense(23, activation='softmax')(x)

model = keras.Model(inputs, outputs)

I0000 00:00:1777403663.845031      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777403663.850967      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 23)             │        29,463 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,079,034 (15.56 MB)

 Trainable params: 29,463 (115.09 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [5]:
labeled = pd.read_csv("/kaggle/input/datasets/jkong05/c-logo-labels/labels.csv")
labeled["sector"] = labeled["sector"].astype("category")
labeled["label_idx"] = labeled["sector"].cat.codes

classes = labeled["sector"].cat.categories
num_classes = len(classes)

train_df, temp_df = train_test_split(labeled, test_size=0.2, stratify=labeled["sector"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["sector"], random_state=42)

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [224, 224])
    
    return tf.cast(img, tf.float32), label

train_data = tf.data.Dataset.from_tensor_slices((train_df["filepath"].values, train_df["label_idx"].values))
train_data = train_data.shuffle(1000).map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)

val_data = tf.data.Dataset.from_tensor_slices((val_df["filepath"].values, val_df["label_idx"].values))
val_data = val_data.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)

test_data = tf.data.Dataset.from_tensor_slices((test_df["filepath"].values, test_df["label_idx"].values))
test_data = test_data.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)

In [9]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', 
             tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top_3_accuracy'),
    ]
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        CHECKPOINT_FILEPATH,
        monitor="val_loss",
        verbose=1,
        save_best_only=True,
        save_weights_only=False,
        mode="min",
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
]

In [10]:
H = model.fit(train_data, validation_data=val_data, epochs=20, callbacks=callbacks)

Epoch 1/20
5126/5127 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.1108 - loss: 3.0243 - top_3_accuracy: 0.2640

2026-04-28 19:24:33.099991: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:24:33.237965: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:24:33.553611: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:24:33.695667: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:24:34.509188: E external/local_xla/xla/stream_

5127/5127 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.1108 - loss: 3.0243 - top_3_accuracy: 0.2640

2026-04-28 19:25:51.217259: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:25:51.359820: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:25:51.709253: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:25:51.851758: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:25:52.634989: E external/local_xla/xla/stream_


Epoch 1: val_loss improved from inf to 2.98957, saving model to /kaggle/working/chkpts/checkpoint.model.keras
5127/5127 ━━━━━━━━━━━━━━━━━━━━ 511s 96ms/step - accuracy: 0.1108 - loss: 3.0243 - top_3_accuracy: 0.2640 - val_accuracy: 0.1196 - val_loss: 2.9896 - val_top_3_accuracy: 0.2792
Epoch 2/20
5125/5127 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.1179 - loss: 3.0069 - top_3_accuracy: 0.2762
Epoch 2: val_loss improved from 2.98957 to 2.98634, saving model to /kaggle/working/chkpts/checkpoint.model.keras
5127/5127 ━━━━━━━━━━━━━━━━━━━━ 191s 37ms/step - accuracy: 0.1179 - loss: 3.0069 - top_3_accuracy: 0.2762 - val_accuracy: 0.1269 - val_loss: 2.9863 - val_top_3_accuracy: 0.2858
Epoch 3/20
5125/5127 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.1195 - loss: 3.0032 - top_3_accuracy: 0.2787
Epoch 3: val_loss improved from 2.98634 to 2.98589, saving model to /kaggle/working/chkpts/checkpoint.model.keras
5127/5127 ━━━━━━━━━━━━━━━━━━━━ 190s 37ms/step - accuracy: 0.1195 - loss: 3.0032 -

In [1]:
score = model.evaluate(test_data)

NameError: name 'model' is not defined

In [2]:
preds = model.predict(dataset).argmax(axis=1)
true_labels = labels 

cm = confusion_matrix(true_labels, preds)
disp = ConfusionMatrixDisplay(cm, display_labels=list(label_to_idx.keys()))

fig, ax = plt.subplots(figsize=(16, 16))
disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.tight_layout()
plt.show()

NameError: name 'model' is not defined